<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w11-mcp-data-apis-third-party/notebook.ipynb)


In [1]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [2]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w11-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 11: Databases, APIs, and third-party servers

**Week 0 · Course B, chapter 3 of 3 · about 60 minutes**

**Goal:** Put a database behind a tool and an API behind another one, without opening either of the two holes this chapter is about. Then look at a server you did not write and decide, on evidence, whether to point a model at it.

**Why it matters:** `cookbook/advanced/08_anti_poisoning.ipynb` treats every ingested spec and every retrieved document as untrusted input. This unit is that rule one level down, at the value a tool was handed. The README's Safety Boundary section says a key never goes in MCP JSON, in a prompt, or in a commit, and exercise 2 is where you find out that a tool's parameter list is one more place it must never go. Session 13 connects to a hosted surface, which is exactly the third-party server lesson 3 asks three questions about.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Both of them **work**. That is the point: the first one answers correctly until somebody sends a quote, and the second one answers correctly while publishing your credential to every client that calls it.

**Offline, and honest about it.** The database is real SQLite, built in a temp directory from unit 9's recorded table of January offsets. `requests` is not installed here, so the API lesson shows the deck's `requests.post` as the source and the notebook's tool does the arithmetic locally: the authentication is the part the exercise is about, and that part is real. The hosted surface in lesson 3 is a **recorded** tool list read from `fixtures/orquestra-tools.json`. Nothing in this notebook touches the network.

## 1. A query that runs, and a hole you can watch open

**Context.** A file gets you a list. A database gets you a question: filter, search and sort
without loading everything, with an index behind it, safe for several callers at once. The deck
adds SQLite to the timezone server for exactly that, and then names the habit that goes with it:
**never string-format a caller's text into SQL.**

Read the connection lifecycle first. One connection, opened at startup and reused by every
handler, closed in the `finally` when the server stops. The `with` block in the cell below is
different: that is setup, it commits and closes, and it is not a handler.

The starter's query works. It answers `Europe` correctly. Then the next cell sends it something
that is not a place name.

**Instructions.**

1. Run the first two cells. `Europe` gives four zones. The second call gives you thirteen, and
   the thirteenth is not a timezone.
2. Fix the query: `LIKE ?`, and hand the value to `execute` as a tuple beside the SQL.
3. Run the second cell again. `Europe` still works. The other one now matches nothing, because
   the database compared it as text instead of parsing it as SQL.

**Expected output**

```
prefix='Europe'
Europe/Berlin
Europe/Lisbon
Europe/London
Europe/Paris

prefix="' UNION SELECT 'pwned' --"
(nothing)
✅ w11-e1 passed
```

Before the fix, that second call ends with a line reading `pwned`.

In [10]:
import json
import sqlite3
import tempfile

WORKDIR = Path(tempfile.mkdtemp(prefix="mcp-unit11-"))
DB_PATH = WORKDIR / "timezones.db"
DB_SERVER_PATH = WORKDIR / "db_server.py"
FIXTURE = (
    REPO_ROOT / "units" / "en" / "unit0" / "w09-mcp-first-server" / "fixtures" / "timezones.json"
)
ZONES = json.loads(FIXTURE.read_text(encoding="utf-8"))["zones"]

# Build the database once, from the fixture. `with` commits and closes it: this
# is setup, not a handler, and the server below owns its own connection.
with sqlite3.connect(DB_PATH) as setup:
    setup.execute("CREATE TABLE locations (timezone TEXT PRIMARY KEY, utc_offset REAL NOT NULL)")
    setup.executemany("INSERT INTO locations VALUES (?, ?)", sorted(ZONES.items()))

DB_SERVER = r'''
"""A timezone converter backed by SQLite, served over MCP."""

import sqlite3

from mcp.server.mcpserver import MCPServer

DB_PATH = r"__DB__"

mcp = MCPServer("Timezone Converter")

# One connection, opened at startup and reused by every handler. check_same_thread
# is off because the handler may run on a worker thread, not the one that opened it.
conn = sqlite3.connect(DB_PATH, check_same_thread=False)
conn.row_factory = sqlite3.Row


@mcp.tool()
def lookup_locations(prefix: str) -> str:
    """Find the timezones whose name contains the given text.

    Args:
        prefix: part of a zone name, for example Europe or Toronto

    Returns:
        The matching zone names, one per line, at most fifty of them.
    """
    sql = "SELECT timezone FROM locations WHERE timezone LIKE ? LIMIT 50"  # <------ EDIT THIS LINE
    try:
        rows = conn.execute(sql, (f"%{prefix}%",)).fetchall()  # <------ EDIT THIS LINE
    except sqlite3.Error as error:
        return f"Error: {error}"
    return "\n".join(row["timezone"] for row in rows)


if __name__ == "__main__":
    try:
        mcp.run(transport="stdio")
    finally:
        conn.close()
'''

DB_SERVER_PATH.write_text(DB_SERVER.replace("__DB__", str(DB_PATH)), encoding="utf-8")
print(f"wrote {DB_SERVER_PATH}")
print(f"{len(ZONES)} rows in {DB_PATH.name}")

wrote /tmp/mcp-unit11-q4wdei2j/db_server.py
12 rows in timezones.db


In [11]:
import asyncio

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

INJECTION = "' UNION SELECT 'pwned' --"


async def call_tool(server_path, name, arguments):
    """Call one tool on one server and return the text it answered with."""
    params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            result = await session.call_tool(name, arguments)
            return result.content[0].text if result.content else ""


async def lookup(prefix):
    return await asyncio.wait_for(
        call_tool(DB_SERVER_PATH, "lookup_locations", {"prefix": prefix}), timeout=20
    )


for prefix in ("Europe", INJECTION):
    print(f"prefix={prefix!r}")
    print(await lookup(prefix) or "(nothing)")
    print()

prefix='Europe'
Europe/Berlin
Europe/Lisbon
Europe/London
Europe/Paris

prefix="' UNION SELECT 'pwned' --"
(nothing)



In [12]:
check("w11-e1", DB_SERVER_PATH)

✅ w11-e1 passed


True

## 2. A credential the client can see

**Context.** The deck's rules for an API key are four lines long and the fourth is the one that
gets broken: a key is read **server-side**, from the environment or a `.env` file. The client
never sends it, never receives it, and never holds it. Not hardcoded, not in a URL, not in a
log, and **not a tool argument or part of a tool result**.

A tool's parameters are its input schema, and the input schema is published to every client that
lists the tools. Put `api_key` in the signature and you have not asked a caller for a
credential, you have told the whole world that this tool takes one, and then a model will
cheerfully put whatever it can find there.

Read the starter's docstring. It documents three arguments. Count the parameters.

**Instructions.**

1. Run both cells. Watch what the schema asks for, and read the end of the result.
2. Take `api_key` out of the signature, and read it from `os.environ` inside the tool instead.
   Set the `Authorization` header only when there is a key: a missing credential is a header you
   leave off, not an exception.
3. Take the key out of the returned string. A tool result goes to the client and into a model's
   context, which is a transcript, which is a log.

**Expected output**

```
the schema wants: ['date_time', 'from_timezone', 'to_timezone']
Time in Asia/Tokyo: 2025-01-21T04:30:00+09:00
the key is in the result: False
✅ w11-e2 passed
```

In [18]:
API_SERVER_PATH = WORKDIR / "api_server.py"

API_SERVER = r'''
"""The deck's API-backed converter, with the credential read server-side."""

import json
import os
from datetime import datetime, timedelta, timezone
from pathlib import Path

from mcp.server.mcpserver import MCPServer

ZONES = json.loads(Path(r"__FIXTURE__").read_text(encoding="utf-8"))["zones"]

mcp = MCPServer("Timezone Converter")

ENDPOINT = "https://api.opentimezone.com/convert"


@mcp.tool()
def convert_timezone(date_time: str, from_timezone: str, to_timezone: str) -> str:  # <------ EDIT THIS LINE
    """Convert a datetime from one timezone to another.

    Args:
        date_time: the datetime in ISO format, for example 2025-01-20T14:30:00
        from_timezone: the source zone, for example America/New_York
        to_timezone: the target zone, for example Asia/Tokyo

    Returns:
        A line naming the target zone and the converted datetime.
    """
    headers = {"Content-Type": "application/json"}
    if api_key := os.environ.get("TIMEZONE_API_KEY"):
        headers["Authorization"] = f"Bearer {api_key}"
    headers["Authorization"] = f"Bearer {api_key}" # <------ EDIT THIS LINE
    payload = {"dateTime": date_time, "fromTimezone": from_timezone, "toTimezone": to_timezone}

    # The deck posts `payload` with `headers` to ENDPOINT here, with timeout=10,
    # raises for status, and reads data["dateTime"] back out of the JSON. requests
    # is not installed in this repo, so the two lines below stand in for that trip.
    # Everything the exercise is about is above them.
    moment = datetime.fromisoformat(date_time).replace(
        tzinfo=timezone(timedelta(hours=ZONES[from_timezone]))
    )
    converted = moment.astimezone(timezone(timedelta(hours=ZONES[to_timezone]))).isoformat()

    return f"Time in {to_timezone}: {converted}"  # <------ EDIT THIS LINE


if __name__ == "__main__":
    mcp.run(transport="stdio")
'''

API_SERVER_PATH.write_text(API_SERVER.replace("__FIXTURE__", str(FIXTURE)), encoding="utf-8")
print(f"wrote {API_SERVER_PATH}")
print("the docstring documents three arguments. Count the parameters.")

wrote /tmp/mcp-unit11-q4wdei2j/api_server.py
the docstring documents three arguments. Count the parameters.


In [19]:
import os

from mcp.client.stdio import get_default_environment

os.environ["TIMEZONE_API_KEY"] = "demo-key-not-a-real-one"

NEW_YORK_TO_TOKYO = {
    "date_time": "2025-01-20T14:30:00",
    "from_timezone": "America/New_York",
    "to_timezone": "Asia/Tokyo",
}


async def call_api_tool(arguments):
    """A stdio child gets the environment you hand it, not the one you have.

    get_default_environment() is a short allowlist: PATH, HOME and a few more.
    Everything else, including a credential, is passed on deliberately or not at all.
    """
    params = StdioServerParameters(
        command=sys.executable,
        args=[str(API_SERVER_PATH)],
        env={**get_default_environment(), "TIMEZONE_API_KEY": os.environ["TIMEZONE_API_KEY"]},
    )
    async with stdio_client(params) as (reader, writer):
        async with ClientSession(reader, writer) as session:
            await session.initialize()
            listed = await session.list_tools()
            wanted = sorted(listed.tools[0].input_schema["properties"])
            print(f"the schema wants: {wanted}")
            # Send what the schema asks for. If it asks for a credential then the
            # client is the one holding it, and that is the whole mistake.
            sending = dict(arguments)
            if "api_key" in wanted:
                sending["api_key"] = os.environ["TIMEZONE_API_KEY"]
            result = await session.call_tool("convert_timezone", sending)
            return result.content[0].text if result.content else ""


try:
    API_RESULT = await asyncio.wait_for(call_api_tool(NEW_YORK_TO_TOKYO), timeout=20)
except Exception as error:
    API_RESULT = f"the call failed: {type(error).__name__}: {error}"

print(API_RESULT)
print(f"the key is in the result: {os.environ['TIMEZONE_API_KEY'] in API_RESULT}")

the schema wants: ['date_time', 'from_timezone', 'to_timezone']
Time in Asia/Tokyo: 2025-01-21T04:30:00+09:00
the key is in the result: False


In [20]:
check("w11-e2", API_SERVER_PATH)

✅ w11-e2 passed


True

## 3. Somebody else's server, and three questions

**Context.** Third-party MCP servers are worth using. You get filesystem or database access
without writing a server, somebody else maintains it, and your client code is the same code
either way. The deck's example is `mcp-open-library`, six tools over the Internet Archive's API,
no key needed.

It also gives you three questions to ask first, and they are the whole lesson:

- **Trust.** The server runs code and may call external APIs. Check those sources.
- **Security.** It needs careful handling of private data and credentials.
- **Surface area.** More tools means more the model can decide to do. Know what is exposed.

This exercise does not install anything. It asks those three questions about a surface you will
actually connect to: the hosted MCP endpoint session 13 uses. The cell below reads a **recorded**
list of its tools from this unit's fixtures, along with the note saying what that recording is
and is not evidence of.

**Instructions.**

1. Run the first cell and read the whole provenance note, including the last line.
2. Answer the three questions in `ANSWERS`, one sentence each, in your own words.
3. The surface-area answer has to name the number of tools. A judgement about surface area
   that does not say how large the surface is has not been made yet.

**Expected output**

```
surface:  https://mcp.geckovision.tech/orquestra/mcp
recorded: 2026-09-03, protocol 2025-11-25
tools (16): start, find_start, list_programs, comprehend_program, ...
✅ w11-e3 passed
```

In [21]:
HOSTED = json.loads(
    (
        REPO_ROOT
        / "units"
        / "en"
        / "unit0"
        / "w11-mcp-data-apis-third-party"
        / "fixtures"
        / "orquestra-tools.json"
    ).read_text(encoding="utf-8")
)
NOTE = HOSTED["_provenance"]

print(f"surface:  {NOTE['surface']}")
print(f"recorded: {NOTE['recorded']}, protocol {NOTE['protocol']}")
print(f"tools ({len(HOSTED['tools'])}): {', '.join(HOSTED['tools'])}")
print()
print(f"is evidence of:     {NOTE['is_evidence_of']}")
print(f"is NOT evidence of: {NOTE['is_not_evidence_of']}")

surface:  https://mcp.geckovision.tech/orquestra/mcp
recorded: 2026-09-03, protocol 2025-11-25
tools (16): start, find_start, list_programs, comprehend_program, prepare_purchase, try_purchase, list_stores, plan_payment, plan_swap, verify_signed_transaction, submit_transaction, read_accounts, prepare_instruction, derive_ata, derive_pda, program_lifecycle

is evidence of:     which tools that surface listed on the day it was read, and nothing about what they do when called
is NOT evidence of: that the surface still lists these sixteen today, or that any of them was called. This is a RECORDED list, not a live claim. Unit 11 uses it to count surface area without asking the network for permission; session 13 connects to the surface itself.


In [22]:
ANSWERS = {  # <------ EDIT THESE THREE LINES
    "trust": "We must verify the server's code sources and the external APIs it calls before granting trust.",
    "security": "Private data and sensitive credentials require rigorous handling to prevent leaks.",
    "surface_area": "The exposure surface consists of 16 tools, which directly dictates the breadth of actions the model can undertake.",
}

for question, answer in ANSWERS.items():
    print(f"{question:>12}  {answer}")

       trust  We must verify the server's code sources and the external APIs it calls before granting trust.
    security  Private data and sensitive credentials require rigorous handling to prevent leaks.
surface_area  The exposure surface consists of 16 tools, which directly dictates the breadth of actions the model can undertake.


In [23]:
check("w11-e3", ANSWERS)

✅ w11-e3 passed


True

## Course B, in three units

| Unit | What it added | The mistake it made visible |
|---|---|---|
| 9 | a tool, a client, one correct call | a relative path, and argument names from memory |
| 10 | a resource, a prompt, a five-step loop | the tool result that never reached the model |
| 11 | a database, an API key, a server you did not write | a query that parses its input, and a schema that publishes your credential |

Three things carry past this course. A tool's signature and docstring are its contract, and
they are read by machines. A value that arrived from outside is data, never code, whether it
lands in a SQL string or in a prompt. And a credential belongs to the server, which means the
client never has to be trusted with it.

Session 13 connects to the hosted surface for real. You have already asked it the three
questions.

Run the scorecard.

In [24]:
review("w11")

w11: 3/3 passed  ·  300/300 marks


True